# Popularity-bucket evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/aellieme/warm_start_diff_models.git
%cd warm_start_diff_models

In [ ]:
%pip install -q -r src/requirements.txt

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import yaml

import pandas as pd

PROJECT_ROOT = Path('/content/warm_start_diff_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/exp_results')
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
KS = [10, 20, 100]

def checkpoint_entry(model, dataset, cli_dataset, maxlen, checkpoint, metadata):
    return {
        'model': model, 'dataset': dataset, 'cli_dataset': cli_dataset,
        'maxlen': maxlen, 'checkpoint': CHECKPOINT_ROOT / model / checkpoint,
        'metadata': CHECKPOINT_ROOT / model / metadata,
    }

def baseline_entry(model, dataset, cli_dataset):
    return {
        'model': model, 'dataset': dataset, 'cli_dataset': cli_dataset,
        'maxlen': None, 'checkpoint': None, 'metadata': None,
    }

MANIFEST = [
    checkpoint_entry('DiffuRec', 'ml-1m', 'ml-1m', 50, 'ml-1m__maxlen_50__seed_42.pt', 'ml-1m__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('DiffuRec', 'ml-1m', 'ml-1m', 100, 'ml-1m__maxlen_100__seed_42.pt', 'ml-1m__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('DiffuRec', 'amazon_baby', 'amazon_Baby', 50, 'amazon_baby__maxlen_50__seed_42.pt', 'amazon_baby__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('DiffuRec', 'amazon_baby', 'amazon_Baby', 100, 'amazon_baby__maxlen_100__seed_42.pt', 'amazon_baby__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('DiffuRec', 'amazon_toys', 'amazon_Toys_and_Games', 50, 'amazon_toys__maxlen_50__seed_42.pt', 'amazon_toys__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('DiffuRec', 'amazon_toys', 'amazon_Toys_and_Games', 100, 'amazon_toys__maxlen_100__seed_42.pt', 'amazon_toys__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'ml-1m', 'ml-1m', 50, 'ml-1m__maxlen_50__seed_42.pth', 'ml-1m__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'ml-1m', 'ml-1m', 100, 'ml-1m__maxlen_100__seed_42.pth', 'ml-1m__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'amazon_baby', 'baby', 50, 'amazon_baby__maxlen_50__seed_42.pth', 'amazon_baby__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'amazon_baby', 'baby', 100, 'amazon_baby__maxlen_100__seed_42.pth', 'amazon_baby__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'amazon_toys', 'toys', 50, 'amazon_toys__maxlen_50__seed_42.pth', 'amazon_toys__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('ADRec', 'amazon_toys', 'toys', 100, 'amazon_toys__maxlen_100__seed_42.pth', 'amazon_toys__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'ml-1m', 'ml-1m', 50, 'ml-1m__maxlen_50__seed_42.pt', 'ml-1m__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'ml-1m', 'ml-1m', 100, 'ml-1m__maxlen_100__seed_42.pt', 'ml-1m__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'amazon_baby', 'amazon_Baby', 50, 'amazon_baby__maxlen_50__seed_42.pt', 'amazon_baby__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'amazon_baby', 'amazon_Baby', 100, 'amazon_baby__maxlen_100__seed_42.pt', 'amazon_baby__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'amazon_toys', 'amazon_Toys_and_Games', 50, 'amazon_toys__maxlen_50__seed_42.pt', 'amazon_toys__maxlen_50__seed_42.metadata.json'),
    checkpoint_entry('SASRec', 'amazon_toys', 'amazon_Toys_and_Games', 100, 'amazon_toys__maxlen_100__seed_42.pt', 'amazon_toys__maxlen_100__seed_42.metadata.json'),
    checkpoint_entry('GPTRec', 'ml-1m', 'ml-1m', 50, 'ml-1m_maxlen50_seed42.pt', 'ml-1m_maxlen50_seed42.yaml'),
    checkpoint_entry('GPTRec', 'ml-1m', 'ml-1m', 100, 'ml-1m_maxlen100_seed42.pt', 'ml-1m_maxlen100_seed42.yaml'),
    checkpoint_entry('GPTRec', 'amazon_baby', 'amazon_Baby', 50, 'amazon_baby_maxlen50_seed42.pt', 'amazon_baby_maxlen50_seed42.yaml'),
    checkpoint_entry('GPTRec', 'amazon_baby', 'amazon_Baby', 100, 'amazon_baby_maxlen100_seed42.pt', 'amazon_baby_maxlen100_seed42.yaml'),
    checkpoint_entry('GPTRec', 'amazon_toys', 'amazon_Toys_and_Games', 50, 'amazon_toys_maxlen50_seed42.pt', 'amazon_toys_maxlen50_seed42.yaml'),
    checkpoint_entry('GPTRec', 'amazon_toys', 'amazon_Toys_and_Games', 100, 'amazon_toys_maxlen100_seed42.pt', 'amazon_toys_maxlen100_seed42.yaml'),
    checkpoint_entry('T-DiffRec', 'ml-1m', 'ml-1m', None, 'ml-1m__maxlen_not_applicable__seed_42.pth', 'ml-1m__maxlen_not_applicable__seed_42.metadata.json'),
    checkpoint_entry('T-DiffRec', 'amazon_baby', 'amazon_Baby', None, 'amazon_baby__maxlen_not_applicable__seed_42.pth', 'amazon_baby__maxlen_not_applicable__seed_42.metadata.json'),
    checkpoint_entry('T-DiffRec', 'amazon_toys', 'amazon_Toys_and_Games', None, 'amazon_toys__maxlen_not_applicable__seed_42.pth', 'amazon_toys__maxlen_not_applicable__seed_42.metadata.json'),
    baseline_entry('TopPopular', 'ml-1m', 'ml-1m'),
    baseline_entry('TopPopular', 'amazon_baby', 'baby'),
    baseline_entry('TopPopular', 'amazon_toys', 'toys'),
    baseline_entry('Random', 'ml-1m', 'ml-1m'),
    baseline_entry('Random', 'amazon_baby', 'baby'),
    baseline_entry('Random', 'amazon_toys', 'toys'),
]

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

assert len(MANIFEST) == 33
assert len({(row['model'], row['dataset'], row['maxlen']) for row in MANIFEST}) == 33

def normalized_dataset(value):
    aliases = {
        'amazon_Baby': 'amazon_baby', 'baby': 'amazon_baby',
        'amazon_Toys_and_Games': 'amazon_toys', 'toys': 'amazon_toys',
    }
    return aliases.get(value, value)

def validate_metadata(row, checkpoint_hash):
    metadata = row['metadata']
    if metadata is None:
        return True, True
    if metadata.suffix == '.json':
        payload = json.loads(metadata.read_text(encoding='utf-8'))
        valid = (
            payload['model'] == row['model'],
            normalized_dataset(payload['dataset']) == row['dataset'],
            payload.get('maxlen') == row['maxlen'],
            payload['seed'] == 42,
            payload['type'] == 'final',
        )
        return all(valid), payload['sha256'] == checkpoint_hash
    payload = yaml.safe_load(metadata.read_text(encoding='utf-8'))
    valid = (
        normalized_dataset(payload['dataset_name']) == row['dataset'],
        payload['dataset']['max_length'] == row['maxlen'],
        payload['seed'] == 42,
        payload['final_train'] is True,
    )
    return all(valid), True

audit_rows = []
for row in MANIFEST:
    checkpoint = row['checkpoint']
    metadata = row['metadata']
    checkpoint_exists = checkpoint is None or checkpoint.is_file()
    metadata_exists = metadata is None or metadata.is_file()
    checkpoint_hash = sha256_file(checkpoint) if checkpoint_exists and checkpoint else '-'
    metadata_valid, stored_hash_matches = (
        validate_metadata(row, checkpoint_hash)
        if metadata_exists else (False, False)
    )
    audit_rows.append({
        'model': row['model'],
        'dataset': row['dataset'],
        'maxlen': row['maxlen'],
        'checkpoint': str(checkpoint) if checkpoint else '-',
        'checkpoint_exists': checkpoint_exists,
        'metadata': str(metadata) if metadata else '-',
        'metadata_exists': metadata_exists,
        'metadata_valid': metadata_valid,
        'sha256': checkpoint_hash,
        'stored_hash_matches': stored_hash_matches,
    })

manifest_audit = pd.DataFrame(audit_rows)
display(manifest_audit)
missing_artifacts = manifest_audit[
    ~manifest_audit['checkpoint_exists']
    | ~manifest_audit['metadata_exists']
    | ~manifest_audit['metadata_valid']
    | ~manifest_audit['stored_hash_matches']
]
display(missing_artifacts)

## Инференс

In [ ]:
def inference_command(row):
    model = row['model']
    dataset = row['cli_dataset']
    checkpoint = str(row['checkpoint']) if row['checkpoint'] else None
    if model == 'DiffuRec':
        return PROJECT_ROOT / 'src/DiffuRec/src', ['python', 'inference.py', '--dataset', dataset, '--max_len', str(row['maxlen']), '--metric_ks', '10', '20', '100', '--device', 'auto', '--checkpoint', checkpoint]
    if model == 'ADRec':
        return PROJECT_ROOT / 'src/ADRec/src', ['python', 'inference.py', '--dataset', dataset, '--max_len', str(row['maxlen']), '--metric_ks', '10', '20', '100', '--checkpoint', checkpoint]
    if model == 'SASRec':
        return PROJECT_ROOT / 'src/SASRec', ['python', 'inference.py', '--dataset', dataset, '--maxlen', str(row['maxlen']), '--metric_ks', '10', '20', '100', '--device', 'auto', '--checkpoint', checkpoint]
    if model == 'GPTRec':
        return PROJECT_ROOT / 'src/GPTRec/src', ['python', 'inference.py', '--dataset', dataset, '--max_len', str(row['maxlen']), '--metric_ks', '10', '20', '100', '--device', 'auto', '--checkpoint', checkpoint, '--decoding_strategy', 'saved']
    if model == 'T-DiffRec':
        return PROJECT_ROOT / 'src/DiffRec/T-DiffRec', ['python', 'inference.py', '--dataset', dataset, '--topN', '[10, 20, 100]', '--device', 'auto', '--checkpoint', checkpoint]
    if model == 'TopPopular':
        return PROJECT_ROOT / 'src/TopPopular', ['python', 'TopPopular_model.py', '--dataset', dataset, '--topk_list', '10', '20', '100']
    if model == 'Random':
        return PROJECT_ROOT / 'src/RandomRecs', ['python', 'RandomRecsModel.py', '--dataset', dataset, '--topk_list', '10', '20', '100']
    raise ValueError(model)

RESULT_PREFIX = 'BUCKET_RESULT_JSON='
RUNNER = PROJECT_ROOT / 'src/research_buckets/run_bucket_inference.py'

def run_one(row, environment):
    cwd, command = inference_command(row)
    script = cwd / command[1]
    runner_command = ['python', str(RUNNER), '--script', str(script), '--', *command[2:]]
    process = subprocess.Popen(runner_command, cwd=cwd, env=environment, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    result_lines = []
    for line in process.stdout:
        if line.startswith(RESULT_PREFIX):
            result_lines.append(line.rstrip())
        else:
            print(line, end='')
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, runner_command)
    if len(result_lines) != 1:
        raise RuntimeError(f'Expected one bucket result, found {len(result_lines)}')
    return json.loads(result_lines[0][len(RESULT_PREFIX):])

RUN_INFERENCE = False
run_results = []
if RUN_INFERENCE:
    if not missing_artifacts.empty:
        raise RuntimeError('Manifest contains missing checkpoints or metadata')
    environment = os.environ.copy()
    environment['EXPERIMENT_OUTPUT_DIR'] = str(RESULTS_ROOT)
    for row in MANIFEST:
        print(row['model'], row['dataset'], row['maxlen'])
        run_results.append((row, run_one(row, environment)))

In [ ]:
rows = []
for manifest_row, bucket_metrics in run_results:
    for bucket, values in bucket_metrics.items():
        for k in KS:
            rows.append({
                'dataset': manifest_row['dataset'],
                'model': manifest_row['model'],
                'maxlen': manifest_row['maxlen'],
                'item_bucket': bucket,
                'K': k,
                'num_cases': values['num_cases'],
                'HR': values['hr'][str(k)],
                'Coverage': values['coverage'][str(k)],
            })

bucket_results = pd.DataFrame(rows)
display(bucket_results)